In [1]:
# # ===== Diagnostic analysis for rule_mapping (paste & run) =====
# import json, math, itertools
# from pathlib import Path
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, classification_report

# ROOT = Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2")
# RULE_OUT_ROOT = ROOT / "outputs" / "rule_mapping"
# PLOT_OUT_ROOT = ROOT / "outputs" / "rule_mapping_diagnostics"
# ensure_dir = lambda p: Path(p).mkdir(parents=True, exist_ok=True) or Path(p)

# ensure_dir(PLOT_OUT_ROOT)

# def plot_and_save_bar(values_dict, title, outpath):
#     # values_dict: {label: value}
#     labels = list(values_dict.keys())
#     vals = [values_dict[l] for l in labels]
#     plt.figure(figsize=(max(6, len(labels)*0.6), 4))
#     sns.barplot(x=labels, y=vals)
#     plt.xticks(rotation=45, ha='right')
#     plt.ylabel(title)
#     plt.ylim(0,1)
#     plt.title(title)
#     plt.tight_layout()
#     plt.savefig(outpath, dpi=200)
#     plt.close()

# def plot_confusion(cm, labels, title, outpath):
#     plt.figure(figsize=(6,5))
#     sns.heatmap(np.array(cm), annot=True, fmt='g', xticklabels=labels, yticklabels=labels, cmap='Blues')
#     plt.xlabel("Predicted")
#     plt.ylabel("True")
#     plt.title(title)
#     plt.tight_layout()
#     plt.savefig(outpath, dpi=200)
#     plt.close()

# langs = sorted([d.name for d in RULE_OUT_ROOT.iterdir() if d.is_dir()])

# report_rows = []
# for LANG in langs:
#     lang_dir = RULE_OUT_ROOT / LANG
#     model_dir = lang_dir / "distil_distilbert-base-multilingual-cased"
#     if not model_dir.exists():
#         print(f"[WARN] no model dir for {LANG} at {model_dir} -> skipping")
#         continue
#     preds_path = model_dir / "rule_mapping_predictions.csv"
#     summary_path = model_dir / "rule_mapping_summary.json"
#     if not preds_path.exists():
#         print(f"[WARN] missing predictions file for {LANG} -> {preds_path}")
#         continue
#     df = pd.read_csv(preds_path)
#     summ = json.load(open(summary_path, encoding="utf8")) if summary_path.exists() else {}

#     # Normalize columns
#     for col in ["gold_label", "final_label", "model_label", "rule_label"]:
#         if col not in df.columns:
#             df[col] = None

#     # Map label strings -> ordered list
#     labels_set = sorted(set(df['gold_label'].dropna().unique().tolist() + df['model_label'].dropna().unique().tolist() + df['final_label'].dropna().unique().tolist()))
#     if not labels_set:
#         print(f"[WARN] No labels detected for {LANG}")
#         continue

#     # Build numeric mapping
#     label_to_idx = {lab:i for i,lab in enumerate(labels_set)}
#     idx_to_label = {i:lab for lab,i in label_to_idx.items()}

#     # Create numeric arrays for metrics
#     mask_gold = df['gold_label'].notnull()
#     if not mask_gold.any():
#         print(f"[WARN] No gold labels for {LANG}; skipping metrics")
#         continue
#     y_true = [label_to_idx.get(x, -1) for x in df.loc[mask_gold, 'gold_label']]
#     y_model = [label_to_idx.get(x, -1) for x in df.loc[mask_gold, 'model_label']]
#     y_rule  = [label_to_idx.get(x, -1) for x in df.loc[mask_gold, 'rule_label']]
#     y_final = [label_to_idx.get(x, -1) for x in df.loc[mask_gold, 'final_label']]

#     # Replace -1 with a fallback (choose 0) to avoid errors; we'll track missing labels separately
#     def safe_replace(arr):
#         return [0 if x==-1 else x for x in arr]
#     y_true_s = safe_replace(y_true); y_model_s = safe_replace(y_model); y_rule_s = safe_replace(y_rule); y_final_s = safe_replace(y_final)

#     # Compute per-class precision/recall/f1 for model, rule, final
#     p_rm = precision_recall_fscore_support(y_true_s, y_model_s, labels=list(range(len(labels_set))), zero_division=0)
#     p_rr = precision_recall_fscore_support(y_true_s, y_rule_s, labels=list(range(len(labels_set))), zero_division=0)
#     p_rf = precision_recall_fscore_support(y_true_s, y_final_s, labels=list(range(len(labels_set))), zero_division=0)

#     perclass_model = {labels_set[i]: {"precision":float(p_rm[0][i]), "recall":float(p_rm[1][i]), "f1":float(p_rm[2][i])} for i in range(len(labels_set))}
#     perclass_rule  = {labels_set[i]: {"precision":float(p_rr[0][i]), "recall":float(p_rr[1][i]), "f1":float(p_rr[2][i])} for i in range(len(labels_set))}
#     perclass_final = {labels_set[i]: {"precision":float(p_rf[0][i]), "recall":float(p_rf[1][i]), "f1":float(p_rf[2][i])} for i in range(len(labels_set))}

#     # Save per-class CSVs
#     pd.DataFrame.from_dict(perclass_model, orient='index').to_csv(model_dir / "diag_model_per_class.csv")
#     pd.DataFrame.from_dict(perclass_rule, orient='index').to_csv(model_dir / "diag_rule_per_class.csv")
#     pd.DataFrame.from_dict(perclass_final, orient='index').to_csv(model_dir / "diag_final_per_class.csv")

#     # Plot per-class F1 comparison (model vs final)
#     f1_model = {lab: perclass_model[lab]['f1'] for lab in labels_set}
#     f1_final = {lab: perclass_final[lab]['f1'] for lab in labels_set}
#     # Dataframe for plotting
#     df_f1 = pd.DataFrame({
#         'label': labels_set,
#         'f1_model': [f1_model[l] for l in labels_set],
#         'f1_final': [f1_final[l] for l in labels_set]
#     })
#     plt.figure(figsize=(max(6,len(labels_set)*0.6),4))
#     x = np.arange(len(labels_set))
#     width = 0.35
#     plt.bar(x - width/2, df_f1['f1_model'], width, label='model_f1')
#     plt.bar(x + width/2, df_f1['f1_final'], width, label='final_f1')
#     plt.xticks(x, labels_set, rotation=45, ha='right')
#     plt.ylim(0,1)
#     plt.legend()
#     plt.title(f"{LANG} per-class F1: model vs final")
#     plt.tight_layout()
#     figpath = PLOT_OUT_ROOT / f"{LANG}_perclass_f1_model_vs_final.png"
#     plt.savefig(figpath, dpi=200); plt.close()

#     # Confusion matrices (model-only vs final)
#     cm_model = confusion_matrix(y_true_s, y_model_s, labels=list(range(len(labels_set))))
#     cm_final = confusion_matrix(y_true_s, y_final_s, labels=list(range(len(labels_set))))
#     plot_confusion(cm_model, labels_set, f"{LANG} Confusion Matrix (model-only)", PLOT_OUT_ROOT / f"{LANG}_cm_model.png")
#     plot_confusion(cm_final, labels_set, f"{LANG} Confusion Matrix (final)", PLOT_OUT_ROOT / f"{LANG}_cm_final.png")

#     # Misclassification table: where final != gold or model != gold. Save top examples.
#     mis_df = df.loc[mask_gold].copy()
#     mis_df['gold_idx'] = y_true_s
#     mis_df['model_idx'] = y_model_s
#     mis_df['final_idx'] = y_final_s
#     mis_df['model_correct'] = mis_df['model_idx'] == mis_df['gold_idx']
#     mis_df['final_correct'] = mis_df['final_idx'] == mis_df['gold_idx']
#     # Cases where final changed the decision relative to model and caused error or fix
#     changed = mis_df[mis_df['model_idx'] != mis_df['final_idx']].copy()
#     changed['outcome'] = changed.apply(lambda r: "fix" if (r['model_correct']==False and r['final_correct']==True) else ("harm" if (r['model_correct']==True and r['final_correct']==False) else "other"), axis=1)
#     changed = changed.sort_values(by=['outcome'])
#     changed.to_csv(model_dir / "diag_changed_by_rule.csv", index=False, encoding="utf8")

#     # Also save all misclassifications where final incorrect
#     mis_all = mis_df[mis_df['final_correct']==False].copy()
#     mis_all.to_csv(model_dir / "diag_final_misclassified.csv", index=False, encoding="utf8")

#     # Print short summary
#     print(f"\nLanguage: {LANG}")
#     print("Labels:", labels_set)
#     print(f"Model accuracy: {summ.get('model_metrics',{}).get('accuracy', np.mean(np.array(y_true_s)==np.array(y_model_s)) if len(y_true_s)>0 else None):.4f}")
#     print(f"Final accuracy: {summ.get('final_metrics',{}).get('accuracy', np.mean(np.array(y_true_s)==np.array(y_final_s)) if len(y_true_s)>0 else None):.4f}")
#     print(f"Saved plots to: {PLOT_OUT_ROOT}")
#     print(f"Saved per-class CSVs under: {model_dir}")

#     # Print 5 sample changed cases for quick inspection (if any)
#     if not changed.empty:
#         print("\nSample cases where rule changed model prediction (up to 5):")
#         sample = changed.head(5)
#         display_cols = ['file_path','gold_label','model_label','final_label','rule_label','rule_confidence','rule_reason']
#         try:
#             print(sample[display_cols].to_string(index=False))
#         except Exception:
#             print(sample[display_cols].head(5).to_dict(orient='records'))

#     report_rows.append({
#         "language": LANG,
#         "n_test": int(len(df)),
#         "n_with_gold": int(mask_gold.sum()),
#         "model_acc": float(summ.get('model_metrics',{}).get('accuracy', math.nan)),
#         "final_acc": float(summ.get('final_metrics',{}).get('accuracy', math.nan))
#     })

# # Save top-level report
# pd.DataFrame(report_rows).to_csv(PLOT_OUT_ROOT / "rule_mapping_overall_report.csv", index=False)
# print("\nDone. Diagnostics saved to:", PLOT_OUT_ROOT)



Language: English
Labels: ['G', 'NC-17', 'PG', 'PG-13', 'R']
Model accuracy: 0.0523
Final accuracy: 0.1628
Saved plots to: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/rule_mapping_diagnostics
Saved per-class CSVs under: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/rule_mapping/English/distil_distilbert-base-multilingual-cased

Sample cases where rule changed model prediction (up to 5):
                                                                                                                                file_path gold_label model_label final_label rule_label  rule_confidence                             rule_reason
                              /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/dataset/English/R_Traffic_2000.txt          R           G           R          R         0.777778 sex_lexical_hits:4;drugs_lexical_hits:7
/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/

In [2]:
# # Aggregation diagnostic cell
# # Paste & run in Colab. Adjust ROOT path if required.
# import json, math, os, sys, tqdm
# from pathlib import Path
# import numpy as np
# import pandas as pd
# from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

# ROOT = Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2")
# DISTILL_OUT = ROOT / "outputs" / "distillation"
# SPLITS_ROOT = ROOT / "data_splits"
# # model id used in pipeline
# MODEL_ID = "distil_distilbert-base-multilingual-cased"

# OUTDIAG = ROOT / "outputs" / "aggregation_diagnostics"
# OUTDIAG.mkdir(parents=True, exist_ok=True)

# LANGS = sorted([d.name for d in SPLITS_ROOT.iterdir() if d.is_dir()])
# print("Languages:", LANGS)

# def load_npz_logits(npz_path):
#     """Load an npz that maps keys->(n_chunks, n_labels) arrays.
#        Return dict: key -> ndarray (n_chunks, n_labels)"""
#     arr = np.load(npz_path, allow_pickle=True)
#     out={}
#     # keys may be saved under array names; try iterator
#     for k in arr.files:
#         out[k] = arr[k]
#     # If npz saved as object arrays mapping, try fallback:
#     if not out:
#         try:
#             obj = np.load(npz_path, allow_pickle=True)
#             if isinstance(obj, np.ndarray) and obj.dtype == object:
#                 # try to interpret as dict
#                 try:
#                     dd = obj.item()
#                     if isinstance(dd, dict):
#                         out = dd
#                 except Exception:
#                     pass
#         except Exception:
#             pass
#     return out

# def safe_get_file_keys(mapping_json, file_path):
#     """Return list of npz keys associated with this file_path using mapping"""
#     # mapping JSON format earlier was: csv_fp -> npz_key (or list)
#     v = mapping_json.get(file_path, None)
#     if v is None:
#         # try basename match
#         base = Path(file_path).name
#         # mapping may store base->key
#         for k, val in mapping_json.items():
#             if Path(k).name == base:
#                 v = val; break
#     return v

# # Aggregation functions
# def agg_mean(ch_logits):
#     return np.mean(ch_logits, axis=0)
# def agg_max(ch_logits):
#     # take chunk with highest max logit
#     idx = np.argmax(ch_logits.max(axis=1))
#     return ch_logits[idx]
# def agg_majority(ch_logits):
#     # per chunk argmax then majority vote; return averaged logits for chosen class
#     argmaxs = np.argmax(ch_logits, axis=1)
#     # if tie use average logits
#     vals,counts = np.unique(argmaxs, return_counts=True)
#     majority_class = int(vals[np.argmax(counts)])
#     logits_mean = ch_logits.mean(axis=0)
#     out = np.zeros_like(logits_mean); out[majority_class]=1.0
#     return out
# def agg_topk_by_student(ch_logits, k=3):
#     # pick k chunks by student confidence (max softmax prob)
#     probs = np.exp(ch_logits - ch_logits.max(axis=1, keepdims=True))
#     probs = probs / probs.sum(axis=1, keepdims=True)
#     conf = probs.max(axis=1)
#     idx = np.argsort(-conf)[:k]
#     return ch_logits[idx].mean(axis=0)
# def agg_topk_by_teacher(ch_logits, teacher_logits_for_chunks, k=3):
#     # pick top-k chunks by teacher confidence (teacher_logits may be None)
#     if teacher_logits_for_chunks is None:
#         return agg_topk_by_student(ch_logits, k=k)
#     t_probs = np.exp(teacher_logits_for_chunks - teacher_logits_for_chunks.max(axis=1, keepdims=True))
#     t_probs = t_probs / t_probs.sum(axis=1, keepdims=True)
#     tconf = t_probs.max(axis=1)
#     idx = np.argsort(-tconf)[:k]
#     return ch_logits[idx].mean(axis=0)
# def agg_weighted_by_teacher(ch_logits, teacher_logits_for_chunks):
#     if teacher_logits_for_chunks is None:
#         return agg_mean(ch_logits)
#     t_probs = np.exp(teacher_logits_for_chunks - teacher_logits_for_chunks.max(axis=1, keepdims=True))
#     t_probs = t_probs / (t_probs.sum(axis=1, keepdims=True)+1e-12)
#     tconf = t_probs.max(axis=1) + 1e-12
#     weights = tconf / tconf.sum()
#     return (ch_logits * weights[:,None]).sum(axis=0)

# # For each language, load necessary artifacts and test several aggregation strategies
# reports = []
# for LANG in LANGS:
#     print("\n---", LANG, "---")
#     splits_dir = SPLITS_ROOT / LANG
#     if not splits_dir.exists():
#         print("no splits for", LANG); continue
#     # load label_map metadata for chunk params
#     label_map_path = splits_dir / "label_map.json"
#     if label_map_path.exists():
#         lm_j = json.load(open(label_map_path, encoding="utf8"))
#         meta = lm_j.get("metadata", {})
#         chunk_max_len = meta.get("chunk_max_len", 256)
#         chunk_stride = meta.get("chunk_stride", 64)
#         print("chunk_max_len:", chunk_max_len, "stride:", chunk_stride)
#     else:
#         chunk_max_len, chunk_stride = 256, 64
#         print("[WARN] no label_map.json; using defaults")

#     test_csv = splits_dir / "test.csv"
#     if not test_csv.exists():
#         print("no test.csv for", LANG); continue
#     df_test = pd.read_csv(test_csv)
#     n_test = len(df_test)
#     # load student chunk logits npz
#     student_npz_path = DISTILL_OUT / LANG / MODEL_ID / "logits" / "student_test_chunk_logits.npz"
#     if not student_npz_path.exists():
#         print("[WARN] student chunk logits missing at", student_npz_path, "-> cannot diag aggregation for", LANG)
#         continue
#     student_npz = load_npz_logits(student_npz_path)
#     print("Loaded student npz keys:", len(student_npz))

#         # load teacher chunk logits (optional)
#     teacher_npz_path = DISTILL_OUT / LANG / ("teacher_" + MODEL_ID) / "logits" / "teacher_test_chunk_logits.npz"
#     # earlier teacher dir might be: outputs/train_pipeline/<Lang>/teacher_<modelid>/logits/teacher_test_chunk_logits.npz
#     alternative_teacher = DISTILL_OUT.parent / "train_pipeline" / LANG / ("teacher_" + MODEL_ID) / "logits" / "teacher_test_chunk_logits.npz"

#     if teacher_npz_path.exists():
#         teacher_npz = load_npz_logits(teacher_npz_path)
#         print("Loaded teacher npz keys:", len(teacher_npz))
#     elif alternative_teacher.exists():
#         teacher_npz = load_npz_logits(alternative_teacher)
#         print("Loaded teacher npz keys (alternative path):", len(teacher_npz))
#     else:
#         teacher_npz = None
#         print("[INFO] teacher chunk logits not found; some strategies will skip teacher weighting")

#     # mapping file_path -> npz key(s) should exist: from earlier sanity check mapping
#     mapping_json_path = DISTILL_OUT / LANG / ("teacher_" + MODEL_ID) / "logits" / "file_path_key_mapping.json"
#     if not mapping_json_path.exists():
#         # fallback to train_pipeline teacher dir mapping
#         alt_map = DISTILL_OUT.parent / "train_pipeline" / LANG / ("teacher_" + MODEL_ID) / "logits" / "file_path_key_mapping.json"
#         if alt_map.exists(): mapping_json_path = alt_map
#     mapping = {}
#     if mapping_json_path.exists():
#         mapping = json.load(open(mapping_json_path, encoding="utf8"))
#         print("Loaded mapping entries:", len(mapping))
#     else:
#         print("[WARN] file_path_key_mapping.json not found: will try basename matching with npz keys")

#     # Predefine aggregation strategies to run
#     strategies = {
#         "mean_all_chunks": lambda ch, tch: agg_mean(ch),
#         "max_chunk": lambda ch, tch: agg_max(ch),
#         "majority_vote": lambda ch, tch: agg_majority(ch),
#         "topk_student_k3": lambda ch, tch: agg_topk_by_student(ch, k=3),
#         "topk_student_k5": lambda ch, tch: agg_topk_by_student(ch, k=5),
#         "topk_teacher_k3": lambda ch, tch: agg_topk_by_teacher(ch, tch, k=3),
#         "weighted_teacher": lambda ch, tch: agg_weighted_by_teacher(ch, tch)
#     }

#     results_per_file = []
#     y_true = []
#     preds = {k: [] for k in strategies.keys()}

#     # Iterate test rows and reconstruct chunk logits for each file
#     for idx, r in df_test.iterrows():
#         fp = str(r.get("file_path"))
#         # gold label mapping: try label_id or label column
#         gold_label = r.get("label")
#         # reconstruct key(s) to fetch from student_npz
#         # mapping may provide a single key or list-of-keys string
#         key_entry = mapping.get(fp, None)
#         keys_for_file = []
#         if key_entry is None:
#             # match by basename
#             base = Path(fp).name
#             for k in student_npz.keys():
#                 if k.endswith(base):
#                     keys_for_file.append(k)
#             # also try direct match
#             if fp in student_npz:
#                 keys_for_file.append(fp)
#         else:
#             # mapping could be string key or list
#             if isinstance(key_entry, list):
#                 keys_for_file = key_entry
#             else:
#                 keys_for_file = [key_entry]

#         # deduplicate
#         keys_for_file = list(dict.fromkeys(keys_for_file))

#         # collect chunk logits arrays: student and teacher aligned arrays
#         student_chunks_arrs = []
#         teacher_chunks_arrs = []
#         for k in keys_for_file:
#             if k in student_npz:
#                 arr = np.array(student_npz[k])
#                 # arr could be shape (n_chunks, n_labels) or maybe (n_labels,) single chunk
#                 if arr.ndim == 1:
#                     arr = arr.reshape(1, -1)
#                 student_chunks_arrs.append(arr)
#             else:
#                 # sometimes keys stored by basename only; try basename match in student_npz
#                 for sk in student_npz.keys():
#                     if Path(sk).name == Path(k).name:
#                         arr = np.array(student_npz[sk])
#                         if arr.ndim ==1: arr = arr.reshape(1,-1)
#                         student_chunks_arrs.append(arr)
#                         break
#             # teacher
#             if teacher_npz is not None:
#                 if k in teacher_npz:
#                     ta = np.array(teacher_npz[k])
#                     if ta.ndim == 1: ta = ta.reshape(1,-1)
#                     teacher_chunks_arrs.append(ta)
#                 else:
#                     # try basename match
#                     for tk in teacher_npz.keys():
#                         if Path(tk).name == Path(k).name:
#                             ta = np.array(teacher_npz[tk])
#                             if ta.ndim == 1: ta = ta.reshape(1,-1)
#                             teacher_chunks_arrs.append(ta)
#                             break

#         # flatten chunks (concatenate arrays)
#         if student_chunks_arrs:
#             ch_logits = np.vstack(student_chunks_arrs)
#         else:
#             # fallback: maybe student npz has key as basename only; attempt direct basename search
#             base = Path(fp).name
#             matched = []
#             for sk in student_npz.keys():
#                 if Path(sk).name == base:
#                     arr = np.array(student_npz[sk])
#                     if arr.ndim == 1: arr = arr.reshape(1,-1)
#                     matched.append(arr)
#             if matched:
#                 ch_logits = np.vstack(matched)
#             else:
#                 # file missing: set zeros
#                 # assume label dimension equals first student's labels
#                 some = next(iter(student_npz.values()))
#                 n_labels = np.array(some).reshape(-1, some.shape[-1]).shape[-1] if hasattr(some, 'shape') else 2
#                 ch_logits = np.zeros((1, n_labels), dtype=float)

#         if teacher_chunks_arrs:
#             tch = np.vstack(teacher_chunks_arrs)
#         else:
#             tch = None

#         # now apply strategies
#         for name, fn in strategies.items():
#             try:
#                 agg_logits = fn(ch_logits, tch)
#             except Exception as e:
#                 # fallback to mean
#                 agg_logits = agg_mean(ch_logits)
#             # convert to probs
#             try:
#                 exps = np.exp(agg_logits - np.max(agg_logits))
#                 probs = exps / (exps.sum()+1e-12)
#             except Exception:
#                 probs = np.ones_like(agg_logits)/len(agg_logits)
#             pred_id = int(np.argmax(probs))
#             preds[name].append(pred_id)

#         # collect true label id if present (try label column or label_id)
#         # we will map string labels to indices later after collecting all possible labels
#         y_true.append(r.get("label"))

#         results_per_file.append({
#             "file_path": fp,
#             "keys_found": keys_for_file,
#             "n_chunks": int(ch_logits.shape[0]),
#             "gold_label": r.get("label"),
#             **{f"pred_{name}": preds[name][-1] for name in strategies.keys()}
#         })

#     res_df = pd.DataFrame(results_per_file)

#     # Need to map label strings to integer ids. Try to read label_map.json to know ordering
#     label_map_p = SPLITS_ROOT / LANG / "label_map.json"
#     label_to_id = {}
#     id_to_label = {}
#     if label_map_p.exists():
#         lmj = json.load(open(label_map_p, encoding="utf8"))
#         label_map = lmj.get("label_map", {})
#         # label_map might be mapping label->int
#         label_to_id = {str(k):int(v) for k,v in label_map.items()}
#         id_to_label = {int(v):str(k) for k,v in label_map.items()}
#     else:
#         # fallback: gather all gold labels and preds and create an index
#         labels = sorted(set([x for x in res_df['gold_label'].unique() if pd.notna(x)]))
#         label_to_id = {lab:i for i,lab in enumerate(labels)}
#         id_to_label = {i:lab for lab,i in label_to_id.items()}

#     # compute metrics for each strategy
#     metrics = {}
#     for name in strategies.keys():
#         ypred = res_df[f"pred_{name}"].tolist()
#         # need y_true numeric
#         y_true_ids = []
#         for g in res_df['gold_label'].tolist():
#             if pd.isna(g):
#                 y_true_ids.append(None)
#             else:
#                 if str(g) in label_to_id:
#                     y_true_ids.append(int(label_to_id[str(g)]))
#                 else:
#                     # sometimes gold label is already id integer
#                     try:
#                         y_true_ids.append(int(g))
#                     except Exception:
#                         y_true_ids.append(None)
#         # filter only where gold present
#         idxs = [i for i,v in enumerate(y_true_ids) if v is not None]
#         if not idxs:
#             metrics[name] = {"n_eval":0}
#             continue
#         y_t = [y_true_ids[i] for i in idxs]
#         y_p = [ypred[i] for i in idxs]
#         acc = accuracy_score(y_t, y_p)
#         f1_macro = f1_score(y_t, y_p, average='macro', zero_division=0)
#         p,r,f,s = precision_recall_fscore_support(y_t, y_p, average=None, zero_division=0)
#         metrics[name] = {"n_eval": len(idxs), "accuracy": float(acc), "f1_macro": float(f1_macro)}

#     # print results summary
#     print("Aggregation results for", LANG)
#     for k,v in metrics.items():
#         print(f"  {k}: n_eval={v.get('n_eval')} acc={v.get('accuracy', math.nan):.4f} f1_macro={v.get('f1_macro', math.nan):.4f}")

#     # save per-file csv and metrics
#     out_csv = OUTDIAG / f"{LANG}_aggregation_predictions.csv"
#     res_df.to_csv(out_csv, index=False, encoding="utf8")
#     out_metrics = OUTDIAG / f"{LANG}_aggregation_metrics.json"
#     json.dump(metrics, open(out_metrics, "w"), indent=2)
#     reports.append((LANG, metrics))

# print("\nAll done. Per-language CSVs + metrics saved to:", OUTDIAG)


Languages: ['English', 'Hindi', 'Marathi']

--- English ---
chunk_max_len: 256 stride: 64
Loaded student npz keys: 172
[INFO] teacher chunk logits not found; some strategies will skip teacher weighting
[WARN] file_path_key_mapping.json not found: will try basename matching with npz keys
Aggregation results for English
  mean_all_chunks: n_eval=172 acc=0.5814 f1_macro=0.5062
  max_chunk: n_eval=172 acc=0.5407 f1_macro=0.2239
  majority_vote: n_eval=172 acc=0.5756 f1_macro=0.5060
  topk_student_k3: n_eval=172 acc=0.5465 f1_macro=0.2761
  topk_student_k5: n_eval=172 acc=0.5523 f1_macro=0.2852
  topk_teacher_k3: n_eval=172 acc=0.5465 f1_macro=0.2761
  weighted_teacher: n_eval=172 acc=0.5814 f1_macro=0.5062

--- Hindi ---
chunk_max_len: 256 stride: 64
Loaded student npz keys: 31
[INFO] teacher chunk logits not found; some strategies will skip teacher weighting
[WARN] file_path_key_mapping.json not found: will try basename matching with npz keys
Aggregation results for Hindi
  mean_all_chunk

In [2]:
# # ---------- Recompute and save full rule-mapping outputs + diagnostics (complete) ----------
# # Paste & run in Colab. Will recreate prediction + summary + diag CSVs and accuracies.csv
# import json, math, sys, time, traceback
# from pathlib import Path
# from collections import defaultdict
# import numpy as np
# import pandas as pd
# from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, accuracy_score

# ROOT = Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2")
# SPLITS_ROOT = ROOT / "data_splits"
# AGG_DIAG = ROOT / "outputs" / "aggregation_diagnostics"
# EXPLAIN_ROOT = ROOT / "outputs" / "explainability"
# RULE_ROOT = ROOT / "outputs" / "rule_mapping"
# MODEL_ID = "distil_distilbert-base-multilingual-cased"

# # Simple rule table + lexical lists (use your existing rule_table.json if present)
# DEFAULT_RULE_TABLE = {
#     "violence": {"threshold": 0.05, "override_label_if_present": {"English": "R", "Hindi": "A", "Marathi": "UA"}},
#     "sex": {"threshold": 0.03, "override_label_if_present": {"English": "NC-17", "Hindi": "A", "Marathi": "UA"}},
#     "drugs": {"threshold": 0.04, "override_label_if_present": {"English": "R", "Hindi": "UA", "Marathi": "UA"}}
# }
# LEXICAL_LISTS = {
#     "violence": ["kill","murder","stab","shoot","blood","attack","fight","rape","gun","knife","die"],
#     "sex": ["sex","sexual","rape","nude","naked","porn","erotic","intimate","adult"],
#     "drugs": ["drug","cocaine","heroin","marijuana","weed","smoke","meth","opioid","alcohol"]
# }

# def ensure_dir(p: Path):
#     p.mkdir(parents=True, exist_ok=True)
#     return p

# def load_label_map(lang: str):
#     p = SPLITS_ROOT / lang / "label_map.json"
#     if not p.exists():
#         raise FileNotFoundError(f"label_map.json missing for {lang} at {p}")
#     jd = json.load(open(p, encoding="utf8"))
#     label_map = {str(k): int(v) for k,v in jd.get("label_map", {}).items()}
#     inv = {int(v): str(k) for k,v in jd.get("label_map", {}).items()}
#     meta = jd.get("metadata", {})
#     return label_map, inv, meta

# def apply_rules_for_file(fp: str, expl_row: dict, tokens_df: pd.DataFrame, raw_text: str, lang: str, rule_table: dict):
#     # Return (rule_label or None, rule_confidence, rule_reason)
#     chosen = None; reasons = []; conf = 0.0
#     # 1) use explainability concept sensitivities if available
#     if expl_row is not None:
#         for cname, cfg in rule_table.items():
#             col = f"concept_{cname}_sensitivity"
#             if col in expl_row and expl_row[col] is not None and not (isinstance(expl_row[col], float) and np.isnan(expl_row[col])):
#                 try:
#                     val = float(expl_row[col])
#                 except Exception:
#                     continue
#                 if val >= float(cfg.get("threshold", 0.05)):
#                     target = cfg.get("override_label_if_present", {}).get(lang, None)
#                     if target:
#                         chosen = target
#                         reasons.append(f"{cname}_sens:{val:.3f}>={cfg.get('threshold')}")
#                         conf = max(conf, min(1.0, val/(cfg.get("threshold",0.05)*4.0)))
#     # 2) token composite scores (if tokens_df provided)
#     if chosen is None and tokens_df is not None and not tokens_df.empty:
#         token_agg = defaultdict(float)
#         for _,r in tokens_df.iterrows():
#             tok = str(r.get("token","")).lower()
#             token_agg[tok] += float(r.get("composite_score",0.0))
#         total = sum(abs(v) for v in token_agg.values()) + 1e-12
#         for cname,cfg in rule_table.items():
#             kws = LEXICAL_LISTS.get(cname, [])
#             ssum = sum(abs(token_agg.get(k,0.0)) for k in kws)
#             frac = ssum/total
#             if frac >= cfg.get("threshold", 0.05):
#                 tgt = cfg.get("override_label_if_present", {}).get(lang, None)
#                 if tgt:
#                     chosen = tgt
#                     reasons.append(f"{cname}_token_frac:{frac:.3f}>={cfg.get('threshold')}")
#                     conf = max(conf, min(1.0, frac/(cfg.get("threshold",0.05)*4.0)))
#     # 3) raw text lexical fallback
#     if chosen is None and raw_text:
#         txt = raw_text.lower()
#         for cname,cfg in rule_table.items():
#             kws = LEXICAL_LISTS.get(cname, [])
#             hits = sum(1 for kw in kws if kw in txt)
#             if hits >= max(1, int(len(kws)*0.15)):
#                 tgt = cfg.get("override_label_if_present", {}).get(lang, None)
#                 if tgt:
#                     chosen = tgt
#                     reasons.append(f"{cname}_lex_hits:{hits}")
#                     conf = max(conf, min(0.7, hits/len(kws)))
#     return chosen, float(conf), ";".join(reasons)

# def combine_model_and_rule(model_label_str, model_label_id, rule_label_str, rule_conf, policy="override_if_confident", threshold=0.6):
#     # Returns final_label_str, final_label_id, source
#     if policy == "model_priority":
#         return model_label_str, model_label_id, "model"
#     if policy == "rule_priority":
#         return (rule_label_str if rule_label_str is not None else model_label_str), (None if rule_label_str is not None else model_label_id), ("rule" if rule_label_str is not None else "model")
#     if rule_label_str is not None and rule_conf >= threshold:
#         return rule_label_str, None, f"rule_{rule_conf:.2f}"
#     return model_label_str, model_label_id, "model"

# # utility: build diag files like earlier
# def compute_per_class_metrics(y_true, y_pred, label_order, label_names):
#     if not y_true:
#         return {}
#     p,r,f,_ = precision_recall_fscore_support(y_true, y_pred, labels=label_order, zero_division=0)
#     per_class = {label_names[i]: {"precision":float(p[i]), "recall":float(r[i]), "f1":float(f[i])} for i in range(len(label_order))}
#     cm = confusion_matrix(y_true, y_pred, labels=label_order).tolist()
#     acc = float(accuracy_score(y_true, y_pred))
#     return {"accuracy": acc, "per_class": per_class, "confusion_matrix": cm}

# # Start processing languages
# langs = sorted([d.name for d in SPLITS_ROOT.iterdir() if d.is_dir()])
# print("Languages:", langs)

# accuracies_rows = []

# for LANG in langs:
#     print("\n===", LANG, "===")
#     try:
#         label_map, inv_label_map, meta = load_label_map(LANG)
#     except Exception as e:
#         print("  Cannot load label_map.json:", e)
#         continue
#     labels = [inv_label_map[i] for i in sorted(inv_label_map.keys())]
#     # aggregation preds
#     agg_csv = AGG_DIAG / f"{LANG}_aggregation_predictions.csv"
#     agg_metrics_json = AGG_DIAG / f"{LANG}_aggregation_metrics.json"
#     if not agg_csv.exists():
#         print("  Missing aggregation CSV:", agg_csv, " — run aggregation diagnostics first. Skipping.")
#         continue
#     df_agg = pd.read_csv(agg_csv)
#     # pick best strategy
#     best_strat = None
#     best_acc = -1.0; best_f1 = -1.0
#     if agg_metrics_json.exists():
#         try:
#             agg_metrics = json.load(open(agg_metrics_json, encoding="utf8"))
#             if isinstance(agg_metrics, dict):
#                 for strat,stats in agg_metrics.items():
#                     acc = stats.get("accuracy", None)
#                     f1 = stats.get("f1_macro", None)
#                     if acc is not None:
#                         if acc > best_acc or (math.isclose(acc, best_acc) and (f1 is not None and f1 > best_f1)):
#                             best_strat = strat; best_acc = acc; best_f1 = f1 if f1 is not None else best_f1
#         except Exception:
#             pass
#     if best_strat is None:
#         # heuristic fallback
#         candidates = [c for c in df_agg.columns if c.startswith("pred_")]
#         pref = ["pred_mean_all_chunks","pred_majority_vote","pred_topk_student_k3","pred_topk_student_k5","pred_max_chunk"]
#         chosen_col = None
#         for p in pref:
#             if p in df_agg.columns:
#                 chosen_col = p; break
#         if chosen_col is None and candidates:
#             chosen_col = candidates[0]
#         if chosen_col is None:
#             print("  No pred_* columns found; cannot proceed for", LANG)
#             continue
#         best_strat = chosen_col.replace("pred_","")
#         print("  No metrics JSON; heuristic selected:", best_strat)
#     print("  Selected agg strategy:", best_strat)

#     # normalize preds -> file-level model_label, model_label_id
#     def normalize(df_preds, st):
#         col = f"pred_{st}"
#         if col not in df_preds.columns:
#             raise ValueError(f"Column {col} not found in agg CSV")
#         out=[]
#         for _,r in df_preds.iterrows():
#             pid = r[col]
#             try:
#                 pid = int(pid)
#             except:
#                 try:
#                     pid = int(float(pid))
#                 except:
#                     pid = None
#             pl = inv_label_map.get(pid, str(pid) if pid is not None else None)
#             out.append({"file_path": r['file_path'], "model_label_id": pid, "model_label": pl})
#         return pd.DataFrame(out)
#     df_preds = normalize(df_agg, best_strat)

#     # load explainability tokens & metrics if available
#     expl_path = EXPLAIN_ROOT / LANG / MODEL_ID / "explainability_metrics_per_file.csv"
#     token_path = EXPLAIN_ROOT / LANG / MODEL_ID / "top_tokens_per_file.csv"
#     df_expl = pd.read_csv(expl_path) if expl_path.exists() else None
#     df_tokens = pd.read_csv(token_path) if token_path.exists() else None

#     # ensure output dir
#     out_dir = ensure_dir(RULE_ROOT / LANG / MODEL_ID)
#     # load or create rule_table.json
#     rule_table_path = out_dir / "rule_table.json"
#     if rule_table_path.exists():
#         rule_table = json.load(open(rule_table_path, encoding="utf8"))
#     else:
#         rule_table = DEFAULT_RULE_TABLE
#         json.dump(rule_table, open(rule_table_path, "w", encoding="utf8"), indent=2)

#     # load test split for gold labels
#     test_csv = SPLITS_ROOT / LANG / "test.csv"
#     df_test = pd.read_csv(test_csv) if test_csv.exists() else pd.DataFrame()

#     rows_out = []
#     # iterate files
#     for _, r in df_preds.iterrows():
#         fp = r['file_path']
#         model_label = r['model_label']
#         model_label_id = r['model_label_id']
#         # gold
#         gold = None
#         if not df_test.empty:
#             q = df_test[df_test['file_path']==fp]
#             if not q.empty:
#                 gold = q.iloc[0].get('label')
#         # expl row
#         expl_row = None
#         if df_expl is not None:
#             m = df_expl[df_expl['file_path']==fp]
#             if m.empty:
#                 base = Path(fp).name
#                 m = df_expl[df_expl['file_path'].str.endswith(base, na=False)]
#             if not m.empty:
#                 expl_row = m.iloc[0].to_dict()
#         # top tokens
#         tokens_df_file = None
#         if df_tokens is not None:
#             m = df_tokens[df_tokens['file_path']==fp]
#             if m.empty:
#                 base = Path(fp).name
#                 m = df_tokens[df_tokens['file_path'].str.endswith(base, na=False)]
#             if not m.empty:
#                 # parse top_tokens field if string
#                 try:
#                     top = m.iloc[0]['top_tokens']
#                     if isinstance(top, str):
#                         top = json.loads(top.replace("'", '"'))
#                     toks = []
#                     for item in top:
#                         if isinstance(item, (list,tuple)) and len(item)>=2:
#                             toks.append({'token': item[0], 'composite_score': float(item[1])})
#                     tokens_df_file = pd.DataFrame(toks)
#                 except Exception:
#                     tokens_df_file = None
#         # raw text
#         raw = ""
#         try:
#             pfp = Path(fp)
#             if pfp.exists():
#                 raw = pfp.read_text(encoding="utf8", errors="ignore")
#         except Exception:
#             raw = ""

#         # apply rules
#         rule_label, rule_conf, rule_reason = apply_rules_for_file(fp, expl_row, tokens_df_file, raw, LANG, rule_table)
#         final_label, final_label_id, decision_source = combine_model_and_rule(model_label, model_label_id, rule_label, rule_conf, policy="override_if_confident", threshold=0.6)
#         # ensure final id
#         if final_label_id is None:
#             if final_label in label_map:
#                 final_label_id = int(label_map[final_label])
#             else:
#                 try:
#                     final_label_id = int(model_label_id) if model_label_id is not None else None
#                 except:
#                     final_label_id = None

#         rows_out.append({
#             "file_path": fp,
#             "gold_label": gold,
#             "model_label": model_label,
#             "model_label_id": int(model_label_id) if model_label_id is not None else None,
#             "rule_label": rule_label,
#             "rule_confidence": float(rule_conf),
#             "rule_reason": rule_reason,
#             "final_label": final_label,
#             "final_label_id": int(final_label_id) if final_label_id is not None else None,
#             "decision_source": decision_source
#         })

#     df_out = pd.DataFrame(rows_out)
#     # save predictions CSV
#     preds_path = out_dir / "rule_mapping_predictions.csv"
#     df_out.to_csv(preds_path, index=False, encoding="utf8")
#     print("  Saved:", preds_path)

#     # compute evaluation summary and per-class CSVs and diag files if gold present
#     if 'gold_label' in df_out.columns and df_out['gold_label'].notnull().any():
#         # map gold to ids using label_map if possible
#         y_true_ids = []
#         for g in df_out['gold_label'].tolist():
#             if pd.isna(g):
#                 y_true_ids.append(None)
#             else:
#                 if str(g) in label_map:
#                     y_true_ids.append(int(label_map[str(g)]))
#                 else:
#                     try:
#                         y_true_ids.append(int(g))
#                     except:
#                         y_true_ids.append(None)
#         model_preds = [int(x) if x is not None else None for x in df_out['model_label_id'].tolist()]
#         final_preds = [int(x) if x is not None else None for x in df_out['final_label_id'].tolist()]
#         idxs = [i for i,v in enumerate(y_true_ids) if v is not None]
#         if idxs:
#             y_t = [y_true_ids[i] for i in idxs]
#             y_m = [model_preds[i] for i in idxs]
#             y_f = [final_preds[i] for i in idxs]
#             label_order = list(range(len(labels)))
#             # compute metrics
#             model_metrics = compute_per_class_metrics(y_t, y_m, label_order, labels)
#             final_metrics = compute_per_class_metrics(y_t, y_f, label_order, labels)
#             # save summary
#             summary = {
#                 "language": LANG,
#                 "model_id": MODEL_ID,
#                 "selected_aggregation_strategy": best_strat,
#                 "n_test_files": len(df_out),
#                 "n_with_gold": len(idxs),
#                 "model_metrics": model_metrics,
#                 "final_metrics": final_metrics
#             }
#             json.dump(summary, open(out_dir / "rule_mapping_summary.json", "w", encoding="utf8"), indent=2)
#             print("  Saved:", out_dir / "rule_mapping_summary.json")
#             # per-class csvs
#             pd.DataFrame.from_dict(model_metrics.get("per_class", {}), orient='index').to_csv(out_dir / "model_per_class.csv")
#             pd.DataFrame.from_dict(final_metrics.get("per_class", {}), orient='index').to_csv(out_dir / "final_per_class.csv")
#             print("  Saved per-class CSVs")
#             # diagnostics: changed_by_rule and final_misclassified (like earlier)
#             df_eval = df_out.copy()
#             df_eval['model_id'] = df_eval['model_label_id']
#             df_eval['final_id'] = df_eval['final_label_id']
#             # build numeric mapping for gold to ids
#             gold_ids = []
#             for g in df_eval['gold_label'].tolist():
#                 if pd.isna(g):
#                     gold_ids.append(None)
#                 else:
#                     if str(g) in label_map:
#                         gold_ids.append(int(label_map[str(g)]))
#                     else:
#                         try:
#                             gold_ids.append(int(g))
#                         except:
#                             gold_ids.append(None)
#             df_eval['gold_id'] = gold_ids
#             df_eval['model_id_num'] = [int(x) if x is not None else None for x in df_eval['model_label_id'].tolist()]
#             df_eval['final_id_num'] = [int(x) if x is not None else None for x in df_eval['final_label_id'].tolist()]
#             df_eval['model_correct'] = df_eval.apply(lambda r: (r['model_id_num'] is not None and r['gold_id'] is not None and r['model_id_num']==r['gold_id']), axis=1)
#             df_eval['final_correct'] = df_eval.apply(lambda r: (r['final_id_num'] is not None and r['gold_id'] is not None and r['final_id_num']==r['gold_id']), axis=1)

#             changed = df_eval[df_eval['model_id_num'] != df_eval['final_id_num']].copy()
#             changed['outcome'] = changed.apply(lambda r: "fix" if (not r['model_correct'] and r['final_correct']) else ("harm" if (r['model_correct'] and not r['final_correct']) else "other"), axis=1)
#             changed.to_csv(out_dir / "diag_changed_by_rule.csv", index=False, encoding="utf8")
#             df_eval[df_eval['final_correct']==False].to_csv(out_dir / "diag_final_misclassified.csv", index=False, encoding="utf8")

#             # per-class diag csvs (also save model/rule/final per-class)
#             pd.DataFrame.from_dict(model_metrics.get("per_class", {}), orient='index').to_csv(out_dir / "diag_model_per_class.csv")
#             pd.DataFrame.from_dict(final_metrics.get("per_class", {}), orient='index').to_csv(out_dir / "diag_final_per_class.csv")
#             # simple rule-only baseline: compute rule_label_id per-file where rule_label exists
#             rule_only_preds = []
#             gold_for_rule = []
#             for _,rr in df_out.iterrows():
#                 if rr['rule_label'] is None:
#                     rule_only_preds.append(None)
#                 else:
#                     rid = label_map.get(rr['rule_label'], None)
#                     rule_only_preds.append(int(rid) if rid is not None else None)
#                 gold_for_rule.append(rr['gold_label'])
#             # compute per-class for rule-only where present
#             # build numeric arrays where both gold and rule exist
#             y_g = []
#             y_r = []
#             for i,gg in enumerate(gold_for_rule):
#                 try:
#                     if gg is None or (pd.isna(gg)):
#                         continue
#                     gid = int(label_map[str(gg)]) if str(gg) in label_map else int(gg)
#                     rid = rule_only_preds[i]
#                     if rid is None:
#                         continue
#                     y_g.append(gid); y_r.append(rid)
#                 except:
#                     continue
#             if y_g:
#                 p,r,f,_ = precision_recall_fscore_support(y_g, y_r, labels=list(range(len(labels))), zero_division=0)
#                 rule_per_class = {labels[i]: {"precision":float(p[i]), "recall":float(r[i]), "f1":float(f[i])} for i in range(len(labels))}
#                 pd.DataFrame.from_dict(rule_per_class, orient='index').to_csv(out_dir / "diag_rule_per_class.csv")
#             else:
#                 # create empty rule_per_class.csv to indicate none
#                 pd.DataFrame().to_csv(out_dir / "diag_rule_per_class.csv")

#             print("  Saved diagnostics CSVs (changed_by_rule, final_misclassified, diag_*).")
#             # append accuracies row for top-level file
#             accuracies_rows.append({
#                 "language": LANG,
#                 "n_test": int(summary.get("n_test_files", len(df_out))),
#                 "n_with_gold": int(summary.get("n_with_gold", len(idxs))),
#                 "selected_agg": best_strat,
#                 "model_acc": model_metrics.get("accuracy", None),
#                 "final_acc": final_metrics.get("accuracy", None)
#             })
#         else:
#             # no idxs with gold
#             json.dump({"language": LANG, "note":"no gold labels present"}, open(out_dir / "rule_mapping_summary.json", "w", encoding="utf8"), indent=2)
#             print("  No gold labels present; saved predictions only.")
#             accuracies_rows.append({
#                 "language": LANG,
#                 "n_test": len(df_out),
#                 "n_with_gold": 0,
#                 "selected_agg": best_strat,
#                 "model_acc": None,
#                 "final_acc": None
#             })
#     else:
#         # no gold_label column found
#         json.dump({"language": LANG, "note":"no gold labels present"}, open(out_dir / "rule_mapping_summary.json", "w", encoding="utf8"), indent=2)
#         print("  No gold labels present; saved predictions only.")
#         accuracies_rows.append({
#             "language": LANG,
#             "n_test": len(df_out),
#             "n_with_gold": 0,
#             "selected_agg": best_strat,
#             "model_acc": None,
#             "final_acc": None
#         })

# # write aggregated accuracies CSV
# acc_out = RULE_ROOT / "accuracies.csv"
# if accuracies_rows:
#     pd.DataFrame(accuracies_rows).to_csv(acc_out, index=False, encoding="utf8")
#     print("\nSaved accuracies summary to:", acc_out)
# else:
#     print("\nNo accuracies rows to save; check earlier logs.")

# print("\n[ALL DONE] Recomputed rule-mapping outputs and diagnostics for languages:", langs)


Languages: ['English', 'Hindi', 'Marathi']

=== English ===
  Selected agg strategy: mean_all_chunks
  Saved: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/rule_mapping/English/distil_distilbert-base-multilingual-cased/rule_mapping_predictions.csv
  Saved: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/rule_mapping/English/distil_distilbert-base-multilingual-cased/rule_mapping_summary.json
  Saved per-class CSVs
  Saved diagnostics CSVs (changed_by_rule, final_misclassified, diag_*).

=== Hindi ===
  Selected agg strategy: mean_all_chunks
  Saved: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/rule_mapping/Hindi/distil_distilbert-base-multilingual-cased/rule_mapping_predictions.csv
  Saved: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/rule_mapping/Hindi/distil_distilbert-base-multilingual-cased/rule_mapping_summary.json
  Saved per-class CSVs
  Save

In [1]:
# Diagnostics & accuracies.csv generator for rule_mapping outputs
from pathlib import Path
import json, pandas as pd, numpy as np

ROOT = Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2")
RULE_ROOT = ROOT / "outputs" / "rule_mapping"
MODEL_ID = "distil_distilbert-base-multilingual-cased"
OUT_ACC = RULE_ROOT / "accuracies.csv"

langs = sorted([d.name for d in (ROOT / "data_splits").iterdir() if d.is_dir()])

rows = []
print("Checking rule-mapping outputs for languages:", langs)
for L in langs:
    folder = RULE_ROOT / L / MODEL_ID
    print(f"\n--- {L} ---\nFolder: {folder}")
    if not folder.exists():
        print("  [MISSING] folder not found.")
        continue

    # list important files
    preds = folder / "rule_mapping_predictions.csv"
    summary = folder / "rule_mapping_summary.json"
    model_per = folder / "model_per_class.csv"
    final_per = folder / "final_per_class.csv"

    for f in [preds, summary, model_per, final_per]:
        print("  ", f.name, ":", "OK" if f.exists() else "MISSING")

    # show sample predictions
    if preds.exists():
        try:
            dfp = pd.read_csv(preds)
            print("  sample predictions (first 5 rows):")
            display(dfp.head(5))
        except Exception as e:
            print("  could not read predictions CSV:", e)

    # load summary if exists
    summary_data = {}
    if summary.exists():
        try:
            summary_data = json.load(open(summary, encoding="utf8"))
            # try to extract key metrics
            model_acc = summary_data.get("model_metrics",{}).get("accuracy", np.nan)
            final_acc = summary_data.get("final_metrics",{}).get("accuracy", np.nan)
            n_test = summary_data.get("n_test_files", None)
            n_with_gold = summary_data.get("n_with_gold", None)
            rows.append({
                "language": L,
                "n_test_files": n_test,
                "n_with_gold": n_with_gold,
                "selected_agg": summary_data.get("selected_aggregation_strategy"),
                "model_acc": model_acc,
                "final_acc": final_acc
            })
            print("  summary loaded. model_acc:", model_acc, "final_acc:", final_acc)
        except Exception as e:
            print("  could not read summary JSON:", e)
    else:
        print("  summary json missing; attempting to compute simple accuracy from predictions CSV (if gold present).")
        if preds.exists():
            try:
                dfp = pd.read_csv(preds)
                if 'gold_label' in dfp.columns and 'model_label_id' in dfp.columns and 'final_label_id' in dfp.columns:
                    # load label_map to map names -> ids, but if numeric ids already present use them
                    valid = dfp[dfp['gold_label'].notnull()].copy()
                    if not valid.empty:
                        # try to coerce gold to ints when possible
                        def safe_label_to_int(x):
                            try:
                                return int(x)
                            except:
                                return None
                        # if gold_label is string labels, we cannot convert without label_map; skip
                        print("  predictions CSV contains gold_label entries but summary not present; skipping auto-accuracy computation (requires label_map).")
                else:
                    print("  predictions CSV does not contain required columns to compute accuracy automatically.")
            except Exception as e:
                print("  error reading preds CSV to compute accuracy:", e)

# write aggregated accuracies CSV
if rows:
    df_acc = pd.DataFrame(rows)
    df_acc.to_csv(OUT_ACC, index=False, encoding="utf8")
    print("\nAggregated accuracies saved to:", OUT_ACC)
    display(df_acc)
else:
    print("\nNo summary rows collected — no accuracies.csv created. Check that rule_mapping_summary.json files exist.")

# Extra: list any per-class CSVs present for quick manual check
print("\nPer-class CSV files present:")
for L in langs:
    p1 = RULE_ROOT / L / MODEL_ID / "model_per_class.csv"
    p2 = RULE_ROOT / L / MODEL_ID / "final_per_class.csv"
    print(L, "model_per_class:", p1.exists(), "final_per_class:", p2.exists())


Checking rule-mapping outputs for languages: ['English', 'Hindi', 'Marathi']

--- English ---
Folder: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/rule_mapping/English/distil_distilbert-base-multilingual-cased
   rule_mapping_predictions.csv : OK
   rule_mapping_summary.json : OK
   model_per_class.csv : OK
   final_per_class.csv : OK
  sample predictions (first 5 rows):


,file_path,gold_label,model_label,model_label_id,rule_label,rule_confidence,rule_reason,final_label,final_label_id,decision_source
0,/content/drive/MyDrive/PhDWorks/4_Final_Writin...,PG-13,R,3,R,0.333333,sex_lexical_hits:2;drugs_lexical_hits:3,R,3,model
1,/content/drive/MyDrive/PhDWorks/4_Final_Writin...,R,PG-13,2,R,0.222222,sex_lexical_hits:1;drugs_lexical_hits:2,PG-13,2,model
2,/content/drive/MyDrive/PhDWorks/4_Final_Writin...,PG-13,R,3,R,0.333333,sex_lexical_hits:3;drugs_lexical_hits:3,R,3,model
3,/content/drive/MyDrive/PhDWorks/4_Final_Writin...,PG,R,3,R,0.333333,sex_lexical_hits:3;drugs_lexical_hits:3,R,3,model
4,/content/drive/MyDrive/PhDWorks/4_Final_Writin...,R,R,3,R,0.555556,sex_lexical_hits:5;drugs_lexical_hits:2,R,3,model


  summary loaded. model_acc: 0.5813953488372093 final_acc: 0.5872093023255814

--- Hindi ---
Folder: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/rule_mapping/Hindi/distil_distilbert-base-multilingual-cased
   rule_mapping_predictions.csv : OK
   rule_mapping_summary.json : OK
   model_per_class.csv : OK
   final_per_class.csv : OK
  sample predictions (first 5 rows):


,file_path,gold_label,model_label,model_label_id,rule_label,rule_confidence,rule_reason,final_label,final_label_id,decision_source
0,/content/drive/MyDrive/PhDWorks/4_Final_Writin...,UA,UA,1,NaN,0.0,NaN,UA,1,model
1,/content/drive/MyDrive/PhDWorks/4_Final_Writin...,A,A,2,NaN,0.0,NaN,A,2,model
2,/content/drive/MyDrive/PhDWorks/4_Final_Writin...,A,U,0,NaN,0.0,NaN,U,0,model
3,/content/drive/MyDrive/PhDWorks/4_Final_Writin...,A,A,2,NaN,0.0,NaN,A,2,model
4,/content/drive/MyDrive/PhDWorks/4_Final_Writin...,A,U,0,NaN,0.0,NaN,U,0,model


  summary loaded. model_acc: 0.8064516129032258 final_acc: 0.8064516129032258

--- Marathi ---
Folder: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/rule_mapping/Marathi/distil_distilbert-base-multilingual-cased
   rule_mapping_predictions.csv : OK
   rule_mapping_summary.json : OK
   model_per_class.csv : OK
   final_per_class.csv : OK
  sample predictions (first 5 rows):


,file_path,gold_label,model_label,model_label_id,rule_label,rule_confidence,rule_reason,final_label,final_label_id,decision_source
0,/content/drive/MyDrive/PhDWorks/4_Final_Writin...,U,U,0,NaN,0.0,NaN,U,0,model
1,/content/drive/MyDrive/PhDWorks/4_Final_Writin...,U,U,0,NaN,0.0,NaN,U,0,model
2,/content/drive/MyDrive/PhDWorks/4_Final_Writin...,U,UA,1,NaN,0.0,NaN,UA,1,model
3,/content/drive/MyDrive/PhDWorks/4_Final_Writin...,UA,U,0,NaN,0.0,NaN,U,0,model
4,/content/drive/MyDrive/PhDWorks/4_Final_Writin...,UA,U,0,NaN,0.0,NaN,U,0,model


  summary loaded. model_acc: 0.5625 final_acc: 0.5625

Aggregated accuracies saved to: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/rule_mapping/accuracies.csv


,language,n_test_files,n_with_gold,selected_agg,model_acc,final_acc
0,English,172,172,mean_all_chunks,0.581395,0.587209
1,Hindi,31,31,mean_all_chunks,0.806452,0.806452
2,Marathi,16,16,mean_all_chunks,0.562500,0.562500



Per-class CSV files present:
English model_per_class: True final_per_class: True
Hindi model_per_class: True final_per_class: True
Marathi model_per_class: True final_per_class: True


In [2]:
# 07_rule_mapping.ipynb — Complete code cell
# Paste and run. (This is self-contained and defensive.)
import json, math, re, time
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import itertools

ROOT = Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2")
SPLITS_ROOT = ROOT / "data_splits"
DISTILL_ROOT = ROOT / "outputs" / "distillation"
CALIB_ROOT = ROOT / "outputs" / "calibration"
EXPLAIN_ROOT = ROOT / "outputs" / "explainability"
RULE_OUT_ROOT = ROOT / "outputs" / "rule_mapping"

STUDENT_MODEL_ID = "distil_distilbert-base-multilingual-cased"

# Default rule thresholds (editable). We'll save to rule_table.json so you can tweak later.
DEFAULT_RULE_TABLE = {
    # concept_name -> (threshold fraction of composite mass to consider "present")
    "violence": {"threshold": 0.05, "override_label_if_present": {"English": None, "Hindi": "A", "Marathi": "UA"}},
    "sex": {"threshold": 0.03, "override_label_if_present": {"English": "NC-17", "Hindi": "A", "Marathi": "UA"}},
    "drugs": {"threshold": 0.04, "override_label_if_present": {"English": "R", "Hindi": "UA", "Marathi": "UA"}}
}
# Fallback lexical lists (if explainability metrics not available)
LEXICAL_LISTS = {
    "violence": ["kill","murder","stab","shoot","blood","attack","fight","rape","gun","knife","die"],
    "sex": ["sex","sexual","rape","nude","naked","porn","erotic","intimate","adult"],
    "drugs": ["drug","cocaine","heroin","marijuana","weed","smoke","meth","opioid","alcohol"]
}

# Helper: ensure directories
def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True); return p

def load_label_map(lang: str):
    path = SPLITS_ROOT / lang / "label_map.json"
    if not path.exists():
        raise FileNotFoundError(f"Missing label_map.json for {lang} at {path}")
    jd = json.load(open(path, encoding="utf8"))
    lm = jd.get("label_map", {})
    # Ensure mapping str->int
    label_map = {str(k): int(v) for k,v in lm.items()}
    inv = {int(v): str(k) for k,v in lm.items()}
    metadata = jd.get("metadata", {})
    return label_map, inv, metadata

def normalize_predictions_df(pred_df: pd.DataFrame, label_map: dict, inv_label_map: dict):
    """
    Normalize prediction DataFrame from distillation outputs into:
    columns: file_path, pred_label (str), pred_label_id (int), probs (numpy array of length num_labels)
    The function is defensive: supports per-class prob columns, or a JSON 'probs' column, etc.
    """
    df = pred_df.copy()
    # ensure file_path exists
    if 'file_path' not in df.columns:
        raise ValueError("predictions.csv missing required column 'file_path'")
    # Try to find probs
    probs_col = None
    # common pattern: columns like prob_0, prob_1..., or 'probs' as stringified list, or class-specific columns
    prob_cols = [c for c in df.columns if re.match(r'^(prob|p_|proba|p)\d*', c, flags=re.I)]
    # try class-name columns
    if not prob_cols:
        # check for label column names matching label_map keys
        for c in df.columns:
            if c in label_map.keys():
                prob_cols.append(c)
    # check for JSON 'probs' column
    if 'probs' in df.columns and df['probs'].notnull().any():
        probs_col = 'probs'
    elif prob_cols:
        probs_col = prob_cols
    else:
        # maybe there are columns like proba_0 ... proba_{n-1}
        numeric_prob_cols = sorted([c for c in df.columns if any(ch.isdigit() for ch in c)])
        if numeric_prob_cols:
            probs_col = numeric_prob_cols

    # build probs_array per row
    num_labels = len(label_map)
    probs_list = []
    pred_label_ids = []
    pred_label_strs = []
    for idx, row in df.iterrows():
        # prefer explicit pred_label/pred_label_id if present
        pl = None
        pli = None
        if 'pred_label' in df.columns and pd.notnull(row.get('pred_label')):
            pl = str(row.get('pred_label'))
        if 'pred_label_id' in df.columns and pd.notnull(row.get('pred_label_id')):
            try: pli = int(row.get('pred_label_id'))
            except Exception: pli = None

        # construct probs
        probs = None
        if probs_col == 'probs':
            try:
                val = row['probs']
                if isinstance(val, str):
                    probs = np.array(json.loads(val))
                elif isinstance(val, (list, tuple, np.ndarray)):
                    probs = np.array(val)
                else:
                    probs = np.array([float(val)])
            except Exception:
                probs = None
        elif isinstance(probs_col, list) and len(probs_col)>0:
            arr=[]
            for pc in probs_col:
                try:
                    v = row.get(pc, 0.0)
                    arr.append(float(v) if pd.notnull(v) else 0.0)
                except Exception:
                    arr.append(0.0)
            probs = np.array(arr)
        else:
            probs = None

        # fallback: one-hot from pred_label_id if available
        if probs is None or len(probs)==0:
            if pli is not None and num_labels>0:
                p = np.zeros((num_labels,), dtype=float)
                if 0 <= pli < num_labels: p[pli]=1.0
                probs = p
            elif pl is not None and pl in label_map:
                p = np.zeros((num_labels,), dtype=float); p[label_map[pl]]=1.0; probs=p
            else:
                # fallback uniform
                probs = np.ones((num_labels,), dtype=float)/max(num_labels,1)

        # ensure length matches
        if len(probs) != num_labels:
            # try to resize (pad/truncate) conservatively
            p2 = np.zeros((num_labels,), dtype=float)
            for i in range(min(len(probs), num_labels)):
                p2[i] = float(probs[i])
            if p2.sum() > 0: p2 = p2 / p2.sum()
            else: p2 = np.ones((num_labels,), dtype=float)/max(num_labels,1)
            probs = p2

        # final pred label if missing
        if pli is None:
            pli = int(np.argmax(probs))
        if pl is None:
            pl = inv_label_map.get(int(pli), str(pli))

        probs_list.append(probs)
        pred_label_ids.append(int(pli))
        pred_label_strs.append(str(pl))

    df['pred_label_id_norm'] = pred_label_ids
    df['pred_label_norm'] = pred_label_strs
    df['pred_probs_norm'] = probs_list
    return df

# Rule engine
def apply_rules_for_file(fp: str, expl_metrics_row: dict, tokens_df: pd.DataFrame, raw_text: str, lang: str, label_map: dict, inv_label_map: dict, rule_table: dict):
    """
    Returns: (rule_label_str, rule_confidence, rule_reason)
    - rule uses concept sensitivities if available, otherwise lexical scan over tokens or raw_text.
    - rule_confidence is a float [0,1] indicating rule certainty.
    """
    # default
    chosen = None
    reason = []
    confidence = 0.0

    # 1) if explanation metrics exist with concept_* columns, use them
    if expl_metrics_row is not None:
        for cname, cfg in rule_table.items():
            col = f"concept_{cname}_sensitivity"
            if col in expl_metrics_row and not pd.isna(expl_metrics_row[col]):
                val = float(expl_metrics_row[col])
                if val >= float(cfg.get("threshold", 0.05)):
                    target = cfg.get("override_label_if_present", {}).get(lang, None)
                    if target:
                        chosen = target
                        reason.append(f"{cname}_sensitivity:{val:.3f} >= {cfg.get('threshold')}")
                        confidence = max(confidence, min(1.0, val / (cfg.get("threshold", 0.05) * 5.0)))  # scaled
    # 2) if tokens_df present, do composite score sum
    if chosen is None and tokens_df is not None and not tokens_df.empty:
        # tokens_df must have 'token' and 'composite_score'
        token_agg = defaultdict(float)
        for idx,r in tokens_df.iterrows():
            token_agg[r['token'].lower()] += float(r.get('composite_score', 0.0))
        total = sum(abs(v) for v in token_agg.values()) + 1e-9
        for cname,cfg in rule_table.items():
            kws = LEXICAL_LISTS.get(cname, [])
            ssum = sum(abs(token_agg.get(k,0.0)) for k in kws)
            frac = ssum / total
            if frac >= cfg.get("threshold", 0.05):
                target = cfg.get("override_label_if_present", {}).get(lang, None)
                if target:
                    chosen = target
                    reason.append(f"{cname}_token_frac:{frac:.3f} >= {cfg.get('threshold')}")
                    confidence = max(confidence, min(1.0, frac / (cfg.get("threshold",0.05)*5.0)))

    # 3) lexical fallback: raw_text search for strong keywords
    if chosen is None and raw_text:
        txt = raw_text.lower()
        for cname,cfg in rule_table.items():
            kws = LEXICAL_LISTS.get(cname, [])
            found = sum(1 for kw in kws if kw in txt)
            if found >= max(1, int(max(1,len(kws))*0.2)):  # rough heuristic
                target = cfg.get("override_label_if_present", {}).get(lang, None)
                if target:
                    chosen = target
                    reason.append(f"{cname}_lexical_hits:{found}")
                    confidence = max(confidence, min(0.8, found/len(kws)))
    return chosen, float(confidence), ";".join(reason)

# Combination: model + rule override
def combine_model_and_rule(model_label_str, model_label_id, model_probs, rule_label_str, rule_confidence, rule_policy="override_if_confident", override_threshold=0.6):
    """
    model_label_str/id: model outputs
    rule_label_str: label string if rule says present else None
    rule_confidence: in [0,1]
    rule_policy:
       - "override_if_confident": if rule_confidence >= override_threshold then rule_label else model_label
       - "model_priority": always model_label
       - "rule_priority": if rule_label exists use it (regardless confidence)
    """
    if rule_policy == "model_priority":
        return model_label_str, model_label_id, "model"
    if rule_policy == "rule_priority":
        if rule_label_str is not None:
            return rule_label_str, None, "rule"
        else:
            return model_label_str, model_label_id, "model"
    # default: override_if_confident
    if rule_label_str is not None and rule_confidence >= override_threshold:
        return rule_label_str, None, f"rule_{rule_confidence:.2f}"
    return model_label_str, model_label_id, "model"

# Metrics utilities
def compute_eval_metrics(y_true, y_pred, labels_list):
    acc = accuracy_score(y_true, y_pred)
    p,r,f,s = precision_recall_fscore_support(y_true, y_pred, labels=labels_list, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=labels_list)
    per_class = {lab: {"precision": float(p[i]), "recall": float(r[i]), "f1": float(f[i])} for i,lab in enumerate(labels_list)}
    return {"accuracy": float(acc), "per_class": per_class, "confusion_matrix": cm.tolist()}

# Main loop across languages
langs = sorted([d.name for d in SPLITS_ROOT.iterdir() if d.is_dir()])
print("Languages found:", langs)

for LANG in langs:
    print("\n=== PROCESS", LANG, "===")
    out_dir = ensure_dir(RULE_OUT_ROOT / LANG / STUDENT_MODEL_ID)
    # load label map
    label_map, inv_label_map, metadata = load_label_map(LANG)
    labels_list = [inv_label_map[i] for i in range(len(inv_label_map))]
    # load test split
    test_csv = SPLITS_ROOT / LANG / "test.csv"
    if not test_csv.exists():
        print(f"[WARN] test.csv missing for {LANG} at {test_csv}; skipping")
        continue
    df_test = pd.read_csv(test_csv)
    # Load student predictions
    pred_path = DISTILL_ROOT / LANG / STUDENT_MODEL_ID / "predictions.csv"
    if not pred_path.exists():
        print(f"[WARN] predictions.csv not found at {pred_path}; trying alternative names...")
        alternatives = list((DISTILL_ROOT / LANG / STUDENT_MODEL_ID).glob("predictions*.csv"))
        if alternatives:
            pred_path = alternatives[0]
            print(f"[INFO] Using alternative: {pred_path}")
        else:
            # cannot proceed without model predictions
            print(f"[ERROR] No predictions file found for {LANG}; skipping")
            continue
    df_pred_raw = pd.read_csv(pred_path)
    # Normalize preds
    df_pred = normalize_predictions_df(df_pred_raw, label_map, inv_label_map)
    # Try to apply calibration if a temperature exists for this model/language
    temp = None
    calib_json = CALIB_ROOT / LANG / STUDENT_MODEL_ID / "calibration_summary.json"
    if calib_json.exists():
        try:
            td = json.load(open(calib_json, encoding="utf8"))
            temp = float(td.get("temperature", td.get("T", 1.0)))
            print(f"[INFO] Loaded temperature T={temp:.3f} from {calib_json}")
        except Exception:
            temp = None
    # If calibration not found, check a saved 'temperature' in calibration folder files
    if temp is None:
        # look for any file with 'temperature' field
        for f in (CALIB_ROOT / LANG / STUDENT_MODEL_ID).glob("*.json"):
            try:
                jd = json.load(open(f, encoding="utf8"))
                if "temperature" in jd:
                    temp = float(jd["temperature"]); break
            except Exception:
                continue
    # Prepare explainability metrics mapping (if available)
    expl_metrics_path = EXPLAIN_ROOT / LANG / STUDENT_MODEL_ID / "explainability_metrics_per_file.csv"
    tokens_path = EXPLAIN_ROOT / LANG / STUDENT_MODEL_ID / "top_tokens_per_file.csv"
    df_expl = pd.read_csv(expl_metrics_path) if expl_metrics_path.exists() else None
    df_tokens = pd.read_csv(tokens_path) if tokens_path.exists() else None

    # Save rule table (create if missing)
    rule_table_path = out_dir / "rule_table.json"
    if not rule_table_path.exists():
        json.dump(DEFAULT_RULE_TABLE, open(rule_table_path, "w", encoding="utf8"), indent=2)
    rule_table = json.load(open(rule_table_path, encoding="utf8"))

    # Prepare final outputs
    rows_out = []
    missing_files = []
    for idx, row in df_test.iterrows():
        fp = row.get("file_path")
        # find model prediction row for this file
        match_row = None
        # exact match first
        matches = df_pred[df_pred['file_path'] == fp]
        if len(matches) == 0:
            # try basename match
            base = Path(str(fp)).name
            matches = df_pred[df_pred['file_path'].str.endswith(base)]
        if len(matches) == 0:
            # try fuzzy by stem
            stem = Path(str(fp)).stem
            matches = df_pred[df_pred['file_path'].str.contains(stem, na=False)]
        if len(matches) >= 1:
            match_row = matches.iloc[0]
        else:
            missing_files.append(fp)
            # create safe defaults
            model_label = None; model_label_id = None; model_probs = np.ones((len(label_map),))/len(label_map)
        if match_row is not None:
            model_label = match_row.get('pred_label_norm')
            model_label_id = int(match_row.get('pred_label_id_norm'))
            probs = match_row.get('pred_probs_norm')
            # model_probs may be stored as list; ensure numpy
            if isinstance(probs, str):
                try: model_probs = np.array(json.loads(probs))
                except Exception: model_probs = np.array(probs.split()).astype(float)
            elif isinstance(probs, (list, tuple, np.ndarray)):
                model_probs = np.array(probs)
            else:
                model_probs = np.array(probs)
            # apply temperature calibration if available
            try:
                if temp is not None and temp > 0:
                    logits = np.log(np.clip(model_probs, 1e-12, 1.0))
                    logits_scaled = logits / temp
                    exps = np.exp(logits_scaled - np.max(logits_scaled))
                    model_probs = exps / exps.sum()
                    model_label_id = int(np.argmax(model_probs))
                    model_label = inv_label_map.get(model_label_id, str(model_label_id))
            except Exception:
                pass

        # get explainability row if available
        expl_row = None
        if df_expl is not None:
            matches_e = df_expl[df_expl['file_path'] == fp]
            if len(matches_e) == 0:
                base = Path(str(fp)).name
                matches_e = df_expl[df_expl['file_path'].str.endswith(base)]
            if len(matches_e) >= 1:
                expl_row = matches_e.iloc[0].to_dict()

        # get tokens_df if available
        tokens_df_for_file = None
        if df_tokens is not None:
            tokens_match = df_tokens[df_tokens['file_path'] == fp]
            if len(tokens_match) == 0:
                base = Path(str(fp)).name
                tokens_match = df_tokens[df_tokens['file_path'].str.endswith(base)]
            if len(tokens_match) >= 1:
                # top_tokens is stored as list-like; may be stringified
                # we expect top_tokens column is either list of tuples or json string
                try:
                    top_tokens = tokens_match.iloc[0]['top_tokens']
                    if isinstance(top_tokens, str):
                        top_tokens = json.loads(top_tokens.replace("'", '"'))
                    # expand into dataframe
                    toks_rows=[]
                    for kv in top_tokens:
                        if isinstance(kv, (list,tuple)) and len(kv)>=2:
                            toks_rows.append({'token':kv[0], 'composite_score':kv[1]})
                    tokens_df_for_file = pd.DataFrame(toks_rows)
                except Exception:
                    tokens_df_for_file = None

        # raw_text fallback (if needed)
        raw_text = ""
        try:
            raw_text = Path(fp).read_text(encoding="utf8", errors="ignore") if fp and Path(fp).exists() else ""
        except Exception:
            raw_text = ""

        # apply rule engine
        rule_label, rule_conf, rule_reason = apply_rules_for_file(fp, expl_row, tokens_df_for_file, raw_text, LANG, label_map, inv_label_map, rule_table)

        # combine using policy
        final_label, final_label_id, decision_source = combine_model_and_rule(
            model_label if model_label is not None else (inv_label_map.get(model_label_id) if model_label_id is not None else None),
            model_label_id,
            model_probs,
            rule_label,
            rule_conf,
            rule_policy="override_if_confident",
            override_threshold=0.6
        )

        # if final_label_id is None (rule override), map label string to id if possible
        if final_label_id is None:
            if final_label in label_map:
                final_label_id = int(label_map[final_label])
            else:
                # fallback map best-effort
                final_label_id = int(model_label_id) if model_label_id is not None else 0

        rows_out.append({
            "file_path": fp,
            "gold_label": row.get("label"),
            "gold_label_id": int(row.get("label_id")) if "label_id" in row else (label_map.get(row.get("label")) if row.get("label") in label_map else None),
            "model_label": model_label,
            "model_label_id": int(model_label_id) if model_label_id is not None else None,
            "model_probs": model_probs.tolist() if hasattr(model_probs, 'tolist') else model_probs,
            "rule_label": rule_label,
            "rule_confidence": rule_conf,
            "rule_reason": rule_reason,
            "final_label": final_label,
            "final_label_id": int(final_label_id),
            "decision_source": decision_source
        })

    df_out = pd.DataFrame(rows_out)
    # compute evaluation (only where gold exists)
    df_eval = df_out[df_out['gold_label'].notnull()].copy()
    if df_eval.empty:
        print(f"[WARN] No gold labels found for {LANG}; skipping evaluation metrics")
    else:
        # map gold_label to ids if needed
        y_true = []
        for idx,r in df_eval.iterrows():
            gl = r['gold_label']
            if gl in label_map:
                y_true.append(int(label_map[gl]))
            else:
                # maybe stored as id already
                try:
                    y_true.append(int(r.get('gold_label_id')))
                except Exception:
                    y_true.append(None)
        y_true = [int(x) if x is not None else 0 for x in y_true]

        # evaluate model-only, rule-only, and final
        model_preds = [int(x) if x is not None else 0 for x in df_eval['model_label_id'].fillna(0).tolist()]
        final_preds = df_eval['final_label_id'].astype(int).tolist()
        # evaluate rule-only mapping: map rule_label -> id if possible
        rule_preds_temp = []
        for r in df_eval.itertuples(index=False):
            rl = r.rule_label
            if rl is None:
                # treat as model fallback
                rule_preds_temp.append(int(r.model_label_id) if not pd.isnull(r.model_label_id) else 0)
            else:
                rule_preds_temp.append(int(label_map.get(rl, r.model_label_id if not pd.isnull(r.model_label_id) else 0)))
        rule_preds = rule_preds_temp

        labels_order = list(range(len(labels_list)))
        # compute metrics
        model_metrics = compute_eval_metrics(y_true, model_preds, labels_order)
        rule_metrics = compute_eval_metrics(y_true, rule_preds, labels_order)
        final_metrics = compute_eval_metrics(y_true, final_preds, labels_order)

        # Save results
        summary = {
            "language": LANG,
            "model_id": STUDENT_MODEL_ID,
            "n_test_files": len(df_test),
            "n_with_gold": len(df_eval),
            "missing_pred_files": missing_files,
            "temperature_used": temp,
            "model_metrics": model_metrics,
            "rule_metrics": rule_metrics,
            "final_metrics": final_metrics
        }
        # write outputs
        out_dir = ensure_dir(RULE_OUT_ROOT / LANG / STUDENT_MODEL_ID)
        df_out.to_csv(out_dir / "rule_mapping_predictions.csv", index=False, encoding="utf8")
        json.dump(summary, open(out_dir / "rule_mapping_summary.json", "w", encoding="utf8"), indent=2)
        # also write a simple metrics CSV per-class
        def perclass_to_df(perclass_dict, labels_list):
            rows=[]
            for i,lab in enumerate(labels_list):
                m = perclass_dict.get(str(i), None) or perclass_dict.get(i, None) or perclass_dict.get(lab, None)
                if m is None:
                    # try summary structure
                    m = {}
                rows.append({"label": lab, "precision": m.get("precision", None), "recall": m.get("recall", None), "f1": m.get("f1", None)})
            return pd.DataFrame(rows)
        pd.DataFrame([{"metric":"model_accuracy", "value":model_metrics['accuracy']},
                      {"metric":"rule_accuracy", "value":rule_metrics['accuracy']},
                      {"metric":"final_accuracy", "value":final_metrics['accuracy']}]).to_csv(out_dir / "accuracies.csv", index=False)
        pd.DataFrame.from_dict(model_metrics['per_class']).T.to_csv(out_dir / "model_per_class.csv")
        pd.DataFrame.from_dict(rule_metrics['per_class']).T.to_csv(out_dir / "rule_per_class.csv")
        pd.DataFrame.from_dict(final_metrics['per_class']).T.to_csv(out_dir / "final_per_class.csv")
        print(f"[OK] Saved rule mapping outputs to {out_dir}")
    # If evaluation not possible, still save df_out
    if df_eval.empty:
        out_dir = ensure_dir(RULE_OUT_ROOT / LANG / STUDENT_MODEL_ID)
        df_out.to_csv(out_dir / "rule_mapping_predictions.csv", index=False, encoding="utf8")
        json.dump({"language": LANG, "note": "no gold labels found; predictions saved"}, open(out_dir / "rule_mapping_summary.json", "w", encoding="utf8"), indent=2)
        print(f"[OK] Saved predictions (no evaluation) to {out_dir}")

print("\n[ALL DONE] Rule mapping for all languages completed.")


Languages found: ['English', 'Hindi', 'Marathi']

=== PROCESS English ===
[OK] Saved rule mapping outputs to /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/rule_mapping/English/distil_distilbert-base-multilingual-cased

=== PROCESS Hindi ===
[OK] Saved rule mapping outputs to /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/rule_mapping/Hindi/distil_distilbert-base-multilingual-cased

=== PROCESS Marathi ===
[OK] Saved rule mapping outputs to /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/rule_mapping/Marathi/distil_distilbert-base-multilingual-cased

[ALL DONE] Rule mapping for all languages completed.
